<a href="https://colab.research.google.com/github/WMFong0/Python-Weather-Report-System/blob/Third-Version-Early-Access/Python_Weather_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
#utility.py

def twentyfourh_to_12h(time: str) -> str:
  time = time.split(':')
  hour = int(time[0])
  if hour >= 12:
    time.append("PM")
    if (hour > 12):
      time[0] = str(hour - 12)
  else:
    time.append("AM")
    if (hour == 0):
      time[0] = str(12)

  return ":".join(time[:-1]) + " " + time[-1]

def No_empty_string(received_data):
  if received_data == "":
    return None
  else: return received_data

District_reference_for_temperature = {
    'Weather_Station': {
        'Tuen Mun': ['N.T.', 'N.T. West', 'Tuen Mun']
    }
}

def select_place(original_selection, raw_data_temperature_datasection: list, variable_name_for_station: str, supported_station: dict = {}):
    # Predefined
    if original_selection in supported_station.keys():
      return District_reference_for_temperature[original_selection]

    available_station = []
    # Fetch available station
    for station_data in raw_data_temperature_datasection:
      available_station.append( station_data[variable_name_for_station] )

    # Remove supported station
    for station in supported_station.values():
      available_station.remove(station)

    # Same Name Case
    if original_selection in available_station:
      return original_selection


    # Required Manual Case
    print("Since we currently doesn't support automatic selection of weather station at your district\n Please select the nearest weather station from the list below: ")
    station = ""

    while True:
      print("Available Station: ")
      for i in range(len(available_station)):
        if (i != 0 and i % 4 == 0):
          print("\t" + available_station[i], end = '\n')
        else:
          print("\t" + available_station[i], end = '')

      station = format_string(input("\nKey in your nearest Weather Station: "))
      # Just in case operation
      if (station in available_station):
        print()
        break
      print("You have input a unavailable station. Please input a available Station." + "\n"*2)

    return station

def lazy_list_message(message_list: list = []):
  for message in message_list:
    message = message.replace('a.m.', 'am').replace('p.m.','pm').replace("No.", "No")
    small_message = [part.strip() for part in message.split('. ') if part.strip()]
    if not small_message:
        continue

    print(f"\t{small_message[0]}")
    for part in small_message[1:]:
      if part != "":
        if ';' in part:
          smaller_part = part.split(';')
          for text in smaller_part:
            print(f"\t\t{text}")
        else:
          print(f"\t\t{part}")

def format_string(string: str = "") -> str:
  string = string.strip()
  splited_string = string.split(" ")
  for i in range(len(splited_string)):
    splited_string[i] = splited_string[i].lower().capitalize()
  return " ".join(splited_string)

def lazy_print_option(message, option_list: list):

  option_list = list(set(option_list))

  if len(option_list) == 0:
    return None
  elif len(option_list) == 1:
    return option_list[0]

  if len(option_list) <= 5:
    for item in option_list:
      if item != option_list[-1]:
        message += (item + ", ")
      else:
        message += item
  else:
    item_index = 0
    message += (("\n") + ("  "))
    for item in option_list:
      item_index += 1
      if item == option_list[-1]:
        message += item
      elif item_index % 4 == 0:
        message += (item + "\n  ")
      else:
        message += (item + ", ")


  while True:
    print("\n" + message)
    user_input = format_string(input(" "))
    if user_input in option_list:
      break
    print("Sorry. You have input a option that is out of the list, please check your input", "\n")

  return user_input

In [19]:
# cwr.py
  # uvindex, temp, humid

import csv
import requests
import json

class Current_Weather_Report():
  raw_data = None
  last_update = None
  avaliable_temperature_station = None
  district = None
  weather_station = None

  def __init__(self) -> None:
    self.raw_data = None
    self.last_update = None
    self.avaliable_temperature_station = []
    self.district = None
    self.weather_station = None
    self.fetch_data()

  def update_user_data(self, user_input:dict) -> None:
    self.district = user_input['District']
    self.weather_station = user_input['Temperature_Weather_Station']

  def fetch_data(self) -> bool:
    try:
      response = requests.get('https://data.weather.gov.hk/weatherAPI/opendata/weather.php?dataType=rhrread&lang=en', timeout = 10)
      response.raise_for_status()

      if not response:
        raise Exception("Received empty data from API")
        return False

      self.raw_data = response.json()

      if not self.raw_data['updateTime']:
        raise Exception("No updateTime. Possible update on Hong Kong Observatory")
        return False

      self.last_update = self.raw_data['updateTime'][11:19]


      self.avaliable_temperature_station = []
      for data in self.raw_data['temperature']['data']:
        self.avaliable_temperature_station.append(data['place'])
      return True

    except Exception as e:
      raise Exception(f"Error getting Current Weather Report data: {str(e)}")

  def fetch_lighting(self) -> dict:


    raw_data_lighting = self.raw_data.get('lightning')

    if (not raw_data_lighting) or raw_data_lighting == "":
      print("Lightning is not happening in Hong Kong right now.\n")
      return

    print(f"Lightning has began at these location,\n"+
          f"starting from {twentyfourh_to_12h(raw_data_lighting['startTime'][11:19])} "+
          f"to {twentyfourh_to_12h(raw_data_lighting['endTime'][11:19])}")

    for temp in raw_data_lighting['data']:
      if temp["occur"] == "true":
        print(f"\t{temp['place']}")

    return raw_data_lighting

  def fetch_uv(self) -> dict:
    try:
      raw_data_uvindex = self.raw_data.get('uvindex')

      if (not raw_data_uvindex) or raw_data_uvindex == "":
        print("UV Index is unavailable at night.\n")
        return None

      raw_data_uvindex = raw_data_uvindex['data'][0] #Only 1 place available for presenting uvindex data

      raw_data_uvindex['recordDesc'] = self.raw_data['uvindex']['recordDesc']
      raw_data_uvindex['updateTime'] = self.raw_data['updateTime'][11:19]

    except Exception as e:
      raise Exception(f"Error getting uvindex: {str(e)}")

    print(f"\n{raw_data_uvindex['recordDesc']} of {twentyfourh_to_12h(raw_data_uvindex['updateTime'])},",
          f"the current uvindex recorded at {raw_data_uvindex['place']} is {raw_data_uvindex['value']}, " +
          f"classified as {raw_data_uvindex['desc']}")


    extra_message = raw_data_uvindex.get("message")
    if extra_message != None and extra_message != "":
      print(f"Here is a special announcement from HKO regarding uv index: \n",
            extra_message)

    print()

    return raw_data_uvindex

  def fetch_temperature(self) -> dict:
    try:
      raw_data_temperature = self.raw_data.get('temperature')

      if (not raw_data_temperature) or raw_data_temperature == "":
        raise Exception(f"Received Null value in temperature. \n"
        "Report this error to the author.")
        return None

      result = {
          'place': self.weather_station,
          'recordTime': raw_data_temperature['recordTime'][11:19]
      }

      for x in raw_data_temperature['data']:
        if x['place'] == result['place']:

          result['value'] = str(x['value']) + " "+ x['unit']

          print(f"At {twentyfourh_to_12h(result['recordTime'])}, in {result['place']},",
          f"the temperature is {result['value']}\n")
          return result

      raise Exception(f"Data not available at requested district")

    except Exception as e:
      raise Exception(f"Error getting uvindex: {str(e)}")

  def fetch_humidity(self) -> dict:
    raw_data_humidity = self.raw_data.get("humidity")
    if raw_data_humidity == None:
      print("Humidity data unavailable")
      return

    temp = raw_data_humidity.get('data')
    if temp == None or temp == []:
      print("Humidity data unavailable. Possible full mainatance of HKO")
      return
    temp = temp[0]

    result = {
        'place': temp['place'],
        'value': str(temp['value']) + " " + temp['unit'],
        'recordTime': raw_data_humidity['recordTime'][11:19]
    }

    print(f"Humidity data recorded at {twentyfourh_to_12h(result['recordTime'])}: \n" +
          f"\tAt {result['place']}, the humudity recorded is {result['value']}.")

    if (temp['unit'] == 'percent'):
      print(f"\t Humidity Level is ", end = "")
      if (temp['value'] < 25):
        print('Low')
      elif (temp['value'] < 75):
        print('Moderate')
      else:
        print('High')

    return result

  def fetch_and_process_warning(self) -> list:
    result = self.raw_data['warningMessage']

    if result == "":
      return None

    print("\nHere "+
          f"{'is 1' if len(result) == 1 else f'are {len(result)}'} warning message"+
          f"{'s' if len(result)!= 1 else ''} from Hong Kong Observatory:")
    lazy_list_message(result)
    return result

  def fetch_and_process_ultil(self) -> dict:
    # Special Weather Tips
    specialWxTips = self.raw_data.get('specialWxTips')
    if specialWxTips != None:
      print("\nHere are some special Weather Tips from HKO: ")
      lazy_list_message(specialWxTips)

    # Tropical Cyclone Position
    tcmessage = self.raw_data.get('tcmessage')
    if (tcmessage != ""):
      print("\nHere are some information relating to Tropical Cyclone from HKO: \n"+
            f"Currently there {'is 1' if len(tcmessage) == 1 else f'are {len(tcmessage)}'} tropical cyclone" +
            f"{'s' if len(tcmessage) != 1 else ''}"+
            "near Hong Kong")

      for typhoon in tcmessage:
        typhoon = typhoon.replace("\n", "\n\t\t")
        print(f"\t{typhoon}")

    # Others are always unavailable without any reason why
    return None

  def fetch_bundle(self) -> bool:
    try:
      # print(self.raw_data)
      self.fetch_lighting()
      self.fetch_uv()
      self.fetch_temperature()
      self.fetch_humidity()
      self.fetch_and_process_warning()
      self.fetch_and_process_ultil()
    except Exception as e:
      raise Exception(f"Error fetching data: {str(e)}")

  # Quick run Current Weather Report
  def run_cwr(self, user_input: dict):
    self.update_user_data(user_input)
    self.fetch_bundle()

In [20]:
# Hourly_Rainfall.py
import requests
import json

PRIORITY_STATIONS = {'Tuen Mun': ["Tuen Mun", "Wetland Park", "Lau Fau Shan" , "Shui Pin Wai"], #屯門，#濕地公園，流浮山，元朗水邊圍
                     'Tin Shui Wai': ["Wetland Park", "Lau Fau Shan", "Shui Pin Wai", "Tuen Mun"], #濕地公園，流浮山，元朗水邊圍，屯門
                     'Yuen Long': ["Shui Pin Wai", "Wetland Park","Shek Kong", "Lau Fau Shan"]} #元朗水邊圍, 濕地公園，石崗，流浮山

# A way to check if the station is still close to the district user mentioend (within 3.0 km)
PRIORITY_STATIONS_bool = {
    'Tuen Mun': [True, False, False, False],
    'Tin Shui Wai': [True, True, True, False],
    'Yuen Long': [True, False, False, False]
}

class Hourly_Rainfall():
  raw_data = None
  processed_data = None
  last_update = None
  district = None
  # Manual approach identifier
  manual = False

  def __init__(self, district: str) -> None:
    self.raw_data = None
    self.processed_data = None
    self.last_update = None
    self.district = district
    self.manual = False

  def fetch_data(self) -> bool:
    try:
      response = requests.get('https://data.weather.gov.hk/weatherAPI/opendata/hourlyRainfall.php?lang=en', timeout = 10)
      response.raise_for_status() # For HTTP error

      if not response:
        raise Exception("Received empty data from API")
        return False

      self.raw_data = response.json()

      '''
      station = []
      for data in self.raw_data['hourlyRainfall']:
        station.append(data['automaticWeatherStation'])
      print(station)
      print(f"Number of station: {len(station)}")
      '''

      if not self.raw_data['obsTime']:
        raise Exception("No obsTime. Possible update on Hong Kong Observatory")
        return False

      self.last_update = self.raw_data['obsTime'][11:19]
      return True

    except Exception as e:
      raise Exception(f"Error getting hourlyRainfall data: {str(e)}")

  # Extract data for all priority stations
  def filter_data(self) -> list:
    # Result = [None, None, None, None] normally
    result = [None for i in range(len(PRIORITY_STATIONS[self.district]))]
    try:

      # If hourlyRainfall doesn't exist on raw_data (the data from API request, raise an exception)
      if not (self.raw_data['hourlyRainfall']):
        raise Exception(f"Possible Full Maintenance on Hong Kong Observatory. ")

        return None

      hourlyRainfall_data = self.raw_data['hourlyRainfall']

      # Manual apporach
      if (self.district not in PRIORITY_STATIONS.keys()):
        self.manual = True
        while True:
          self.district = select_place(self.district, hourlyRainfall_data, 'automaticWeatherStation', [])
          for data in hourlyRainfall_data:
            if (data['automaticWeatherStation'] == self.district):
              if (data['value'] != 'M'):
                print(f"During last hour of {self.last_update}, in {data[0]}, the rainfall amount is {data[1]}\n")
                return
              else:
                print("This weather station is under maintenance. Please select another station")
                break

        return None

      for data in hourlyRainfall_data:
        if ((data['automaticWeatherStation'] in PRIORITY_STATIONS[self.district]) and data['value'] != 'M'):
          index = PRIORITY_STATIONS[self.district].index(data['automaticWeatherStation']) # as index will raise exception if it doesn't find its answer, very dumb feature
          result[index] = [data['automaticWeatherStation'] , data['value'] + " " + data['unit']]

      if (result != [None, None, None, None]):
        self.processed_data = result
        return result
      else: return None # return none if all 4 of them is not working


    except Exception as e:
      raise Exception(f"Error filtering hourlyRainfall data: {str(e)}")

  def get_nearest_rainfall_data(self):

    for i in range(len(self.processed_data)):
      nearest_rainfall_data = self.processed_data[i]
      if (nearest_rainfall_data):

        if (PRIORITY_STATIONS_bool[self.district][i] == False):
          print("As all the automatic weather station next to you is under maintenance, we will present data from other weather station.\n"
          + "The data retrived might be less accurate. We are sorry for such inconvenience.")

        print(f"During last hour of {self.last_update}, in {nearest_rainfall_data[0]}, the rainfall amount is {nearest_rainfall_data[1]}\n")
        return

    print(f"All 4 nearest automatic weather staion is not available/ under maintenance. Please try again later.") # This shouldn't be used normally. Just a redundant command for just incase.
    return

def run_Hourly_Rainfall(district: str):

  rainfall = Hourly_Rainfall(district)
  try:
      if not (rainfall.fetch_data()):
        raise Exception("Hong Kong Observary API is currently unavailable")

      if (rainfall.filter_data() == None):
        raise Exception("All 4 nearest automatic weather staion is not available/ under maintenance. Please try again later.")

      if (rainfall.manual == False):
        rainfall.get_nearest_rainfall_data()

  except Exception as e:
      print(f"Error: {str(e)}")


In [21]:
# User_Selection.py
import csv
database_reference = {
    'Area' : ['Kowloon', 'New Territories', 'Hong Kong Island'],
    'District' : ['Central & Western District', 'Eastern District', 'Kwai Tsing', 'Islands District', 'North District', 'Sai Kung', 'Sha Tin', 'Southern District', 'Tai Po', 'Tsuen Wan', 'Tuen Mun', 'Wan Chai', 'Yuen Long', 'Yau Tsim Mong', 'Sham Shui Po', 'Kowloon City', 'Wong Tai Sin', 'Kwun Tong'],
    'Temperature_Weather_Station' : {}
}


def fetching():
  reader = csv.DictReader(open('Weather_Station_for_temp.csv', mode = 'r', encoding = 'utf-8-sig'))

  for row in reader:
    database_reference['Temperature_Weather_Station'] [row['Weather Station for Temperature']] = [row['Area'], row['Location at Area'], row['District'], row['Location in District']]

def validate_temperature_station(avaliable_temperature_station):

  # Quick validation
  if len(database_reference['Temperature_Weather_Station'].keys()) == len(avaliable_temperature_station):
    return

  # Unmatch list length, begin deeper checking
  for station in database_reference['Temperature_Weather_Station']:
    if station not in avaliable_temperature_station:
      database_reference['Temperature_Weather_Station'].pop("station")

  # Double Validation
  if len(database_reference['Temperature_Weather_Station'].key()) == len(avaliable_temperature_station):
    return

  else:
    print("Error during validation of temperature station")
    return


#Check what did user input first
def user_selection():
  user_input = format_string(input("Please key in your nearest location (area, district, weather station): "))

  # if user input a area
  if user_input in database_reference['Area']:
    # ask user to select the district inside area
    return district_selection(user_input)
  elif user_input in database_reference['District']:
    # Confirms that only one weather station in the refered district
    return temperature_weather_station_selection(user_input)
  else:
    # ask user to select the area first
    return area_selection()

def area_selection():
  while True:
    print("Sorry. We were unable to identify your selection.")
    area_input = lazy_print_option("Please key in your area below.\nE.g. ", database_reference['Area'])


    if area_input in database_reference['Area']:
      break
  return district_selection(area_input)

def district_selection(area_input):
  district_filtered = []
  print(f"As you have key in {area_input} in the last prompt. We will filter out the district inside {area_input}")


  option = lazy_print_option(message='Would you like to further filter the district by their location at area?\nAccepted Option: ', option_list= ["Yes", "No"])

  if (option == "Yes"):
    district_filtered = district_direction_selection(area_input)

  else:
    for index in database_reference['Temperature_Weather_Station']:
      data = database_reference['Temperature_Weather_Station'][index]
      if data[0] == area_input:
        if data[2] not in district_filtered:
          district_filtered.append(data[2])

  district_input = lazy_print_option("Please select your nearest district: ", district_filtered)

  return temperature_weather_station_selection(district_input)


def district_direction_selection(area_input):
  available_direction = {}  # Direction: [District1, District2, ...]

  for index in database_reference['Temperature_Weather_Station']:
    data = database_reference['Temperature_Weather_Station'][index]
    if data[0] != area_input:
      continue

    if data[1] not in available_direction:
      available_direction[data[1]] = [data[2]]
    else:
      if data[2] not in available_direction[data[1]]:
        available_direction[data[1]].append(data[2])

  user_select = lazy_print_option(message=f"Select your direction from the center of {area_input}: ", option_list=list(available_direction.keys()))

  return available_direction[user_select]

def temperature_weather_station_selection(district_input):
  available_weather_stations, Output = {}, {'District': None, 'Temperature_Weather_Station': None}

  for weather_station in database_reference['Temperature_Weather_Station']:
    data = database_reference['Temperature_Weather_Station'][weather_station]
    if data[2] != district_input:
      continue

    # As data[3] is blank, that means its the only weather station in district
    if data[3] == '':
      Output['District'],Output['Temperature_Weather_Station'] = weather_station, weather_station
      return Output

    # As data[3] is not blank
    # There's only one option in every section
    available_weather_stations[data[3]] = weather_station

  while True:
    print('\n' + "Please select the nearest weather_station (Input the name of the weather station or the direction)")
    for direction in available_weather_stations:
      weather_station = available_weather_stations[direction]
      print(f"\t[{district_input} {direction}] {weather_station} ")
    weather_station_input = format_string(input())
    if weather_station_input in available_weather_stations.keys() or weather_station_input in available_weather_stations.values():
      break
    print("Invalid input. Only Input the name of the weather station or the direction")

  Output['District'],Output['Temperature_Weather_Station'] = district_input, weather_station_input if weather_station_input in available_weather_stations.values() else available_weather_stations[weather_station_input]
  return Output



def user_selection_bundle(avaliable_temperature_station: list):
  fetching()
  validate_temperature_station(avaliable_temperature_station)
  return user_selection()

In [25]:
# Main.py
import requests
import json
import datetime

current_datetime = datetime.datetime.now().astimezone(datetime.timezone(datetime.timedelta(hours=8))); # Enforce Hong Kong Timezone

cwr = Current_Weather_Report()

print("Welcome using Weather Report System." + "\n" +
      "This system uses Hong Kong Observatory Data to report")

user_input = user_selection_bundle(cwr.avaliable_temperature_station)

print("\n" + "="*(20+len(f" Weather Report in {user_input['District']} ")+20) + "\n" + "="*20 + f" Weather Report in {user_input['District']} " + "="*20 + "\n")

print(current_datetime.strftime('Today is %Y/%m/%d. \nCurrent Time: %H:%M:%S') + "\n")

run_Hourly_Rainfall(user_input['District'])
cwr.run_cwr(user_input)




Welcome using Weather Report System.
This system uses Hong Kong Observatory Data to report
Please key in your nearest location (area, district, weather station): Hong KOng
Sorry. We were unable to identify your selection.

Please key in your area below.
E.g. Kowloon, Hong Kong Island, New Territories
 NeW TeRriToRies
As you have key in New Territories in the last prompt. We will filter out the district inside New Territories

Would you like to further filter the district by their location at area?
Accepted Option: No, Yes
 Yes

Select your direction from the center of New Territories: South, East, West, North
 South

Please select the nearest weather_station (Input the name of the weather station or the direction)
	[Islands District East] Cheung Chau 
	[Islands District West] Chek Lap Kok 
West

==================== Weather Report in Islands District ====================

Today is 2025/08/22. 
Current Time: 02:01:20

Error: 'Islands District'
Lightning is not happening in Hong Kong rig